# Qwen Image Edit Web Server (Nunchaku) — Paperspace

nunchaku版 Qwen-Image-Edit-Rapid を Flask Web サーバーとして起動し、cloudflared で外部公開します。

**必要環境:**
 - nunchaku導入済Docker `shibashiba2/paperspace-gradient-base-py314-pytorch2.10:v1.0` 等
 - free-A4000以上

## 1. torchとnunchakuのインストール(インストールされていない場合のみ)

In [ ]:
!pip install -U requests

import os
import sys
import subprocess
import platform

def get_system_cuda_version():
    """nvidia-smiを使用してシステムのCUDAバージョンを取得する"""
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits"]).decode()
        # CUDAバージョンを直接取得する（もしくはnvccから取得）
        cuda_out = subprocess.check_output(["nvcc", "--version"]).decode()
        for line in cuda_out.split('\n'):
            if "release" in line:
                return line.split("release ")[1].split(",")[0].replace(".", "")
    except:
        # 万が一取得失敗した場合はデフォルトを指定（例: 130）
        return "130"
    return "130"

def run_command(command):
    print(f"Executing: {command}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + command.split())

# 1. CUDAバージョンの取得
cuda_tag = f"cu{get_system_cuda_version()}"
print(f"🚀 Detected System CUDA: {cuda_tag}")

# 2. PyTorchのインストール (CUDAバージョンに合わせる)
print(f"📦 Installing PyTorch for {cuda_tag}...")
torch_index_url = f"https://download.pytorch.org/whl/{cuda_tag}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch", "torchvision", "--index-url", torch_index_url])

# 3. Nunchakuの適切なWheelを探す
import torch
import requests
import re

def get_nunchaku_url(repo="nunchaku-ai/nunchaku"):
    if not torch.cuda.is_available():
        print("⚠️ CUDA is not available.")
        return None

    # 1. 環境情報の取得と正規化
    # Python: cp312
    py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
    
    # Torch: 2.10.0 -> 2.10
    torch_raw = torch.__version__.split('+')[0]
    torch_parts = torch_raw.split('.')
    torch_ver = f"{torch_parts[0]}.{torch_parts[1]}" # メジャー.マイナーのみ
    
    # CUDA: 12.8 -> 12.8 (そのまま)
    cuda_ver = torch.version.cuda
    # もしドットがない形式(121など)が混在する場合に備え、ドットありを基本にする
    if '.' not in cuda_ver:
        cuda_ver = f"{cuda_ver[:-1]}.{cuda_ver[-1:]}" 

    plat = "linux_x86_64" if platform.system().lower() == "linux" else "win_amd64"
    
    print(f"🔍 Searching for: Py={py_ver}, Torch={torch_ver}, CUDA={cuda_ver}, Plat={plat}")

    # 2. GitHub APIでAssetsを取得
    try:
        api_url = f"https://api.github.com/repos/{repo}/releases/latest"
        assets = requests.get(api_url, timeout=10).json().get("assets", [])
    except Exception as e:
        print(f"❌ API Error: {e}")
        return None

    # 3. 命名規則に基づいたマッチング
    for asset in assets:
        name = asset["name"]
        if not name.endswith(".whl"):
            continue
            
        # Nunchaku特有の形式 "cu12.8torch2.10" を構成
        target_suffix = f"cu{cuda_ver}torch{torch_ver}"
        
        # 条件判定
        if (target_suffix in name) and (py_ver in name) and (plat in name):
            return asset["browser_download_url"]
            
    return None

# 4. Nunchakuのインストール
nunchaku_url = get_nunchaku_url()
if nunchaku_url:
    print(f"✅ Found Nunchaku Wheel: {nunchaku_url}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", nunchaku_url])
    print("\n🎉 All installations completed successfully!")
else:
    print("\n❌ 条件に合うNunchakuのバイナリが見つかりませんでした。")
    print("GitHubのReleasesを確認し、手動でビルドが必要な可能性があります。")

## 2. パッケージインストール

In [ ]:
%cd /tmp

!apt -y remove python3-blinker
!pip install "diffusers>=0.36.0,<0.37.0"
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130 --force-reinstall
!pip install "flask>=3.0" pillow huggingface_hub psutil transformers accelerate safetensors googletrans

!wget -qN https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("OK")

## 3. HuggingFace ログイン（任意）

ログインするとモデルダウンロードが高速化されます（レート制限緩和）。  
`HF_TOKEN` を登録しておくと自動認証されます。 
スキップも可。

In [ ]:
import os
from getpass import getpass

os.environ['HF_HOME'] = "/tmp/hf_hub/"

try:
    hf_token = os.environ.get("HF_TOKEN") or getpass("Enter HF token: ")
    from huggingface_hub import login
    login(token=hf_token)
    print("HuggingFace: HF_TOKEN で認証しました")
except Exception:
    print("HF_TOKEN が未登録です。手動ログインを試みます...")
    print("スキップする場合はこのセルの出力を無視してください。")
    try:
        from huggingface_hub import login
        login()
    except Exception:
        print("HuggingFace: 未ログイン（匿名ダウンロード）")

## 4. app_nunchaku.py を配置

In [ ]:
import shutil

!git clone https://github.com/id-fa/simple-image-edit-with-qwen.git repo_tmp
!cp repo_tmp/server/app_nunchaku.py .
!cp -r repo_tmp/server/lib .
%mkdir tmp

print("準備完了")
!ls -la app_nunchaku.py nunchaku_lora_qwen.py 2>/dev/null || echo "⚠ ファイルが見つかりません。"

## 5. LoRAのダウンロード（任意）

LoRAはサンプルです。任意のLoRAをダウンロードするなりアップロードするなりGoogleDriveからコピーするなりしてください。

In [ ]:
!hf download sbeland/qwen2512-loras --local-dir /tmp/LoRA

print("OK")

## 6. 設定

パスワードやプリセットを変更できます。
LoRAダウンロードをスキップしていたら `LORAS = []` に変えてください。

In [ ]:
# --- 設定 ---
PASSWORD = "password"       # 生成パスワード
PORT = 5000                 # サーバーポート
GALLERY = True              # ギャラリーモード有効化

# プリセット (空リストならボタン非表示)
PRESETS = [
    "高画質化::Enhance quality and fix artifacts.",
    "テキスト除去::Remove all overlaid text, subtitles, and credits.",
    "画像1に画像2の服を着せる::将 Image-1 中的角色穿上 Image-2 的服装",
]

# LoRA
LORAS = [
    "/tmp/LoRA/gnass_2512_qwen.safetensors",
]

print("OK")

## 7. サーバー起動 + cloudflared 公開

Flask サーバーをバックグラウンドで起動し、cloudflared で公開URLを生成します。
モデルのダウンロードがあるので時間が掛かります。
出力に表示される `https://******.trycloudflare.com` のURLにアクセスしてください。

In [ ]:
import subprocess, time, re, threading

SERVER_LOG = "/tmp/server.log"

# コマンドライン引数を構築
cmd = [
    "python", "-u", "./app_nunchaku.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--password", PASSWORD,
    "--steps", "8",
    "--rank", "128",
    "--num-blocks-on-gpu","2",
]
if GALLERY:
    cmd.append("--gallery")
for lora in LORAS:
    cmd.extend(["--lora", lora])
for preset in PRESETS:
    cmd.extend(["--preset", preset])

print(f"starting server: {' '.join(cmd)}")

# ログファイルに出力（パイプバッファ詰まり防止）
log_fh = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

# ログを監視してサーバー起動完了を待機
print("waiting for model to load...")
ready = False
deadline = time.time() + 600  # 10分タイムアウト
seen = 0
while time.time() < deadline:
    time.sleep(2)
    if server_proc.poll() is not None:
        print("ERROR: server process exited unexpectedly")
        with open(SERVER_LOG) as f:
            print(f.read())
        break
    with open(SERVER_LOG) as f:
        lines = f.readlines()
    # 新しい行を表示
    for line in lines[seen:]:
        print(line, end="")
    seen = len(lines)
    if any("server starting at" in l for l in lines):
        ready = True
        break

if not ready:
    raise RuntimeError("Server failed to start. Check server.log")

print(f"\nserver is running on port {PORT}")
print(f"starting cloudflared tunnel...")

# cloudflared 起動 (stderr にURL出力)
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

# stderr を読み続けるスレッド（バッファ詰まり防止）
tunnel_lines = []
def drain_tunnel_stderr():
    for line in iter(tunnel_proc.stderr.readline, ""):
        tunnel_lines.append(line)
threading.Thread(target=drain_tunnel_stderr, daemon=True).start()

# URL抽出を待機
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    time.sleep(1)
    for line in tunnel_lines:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            public_url = m.group(1)
            break
    if public_url:
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  Public URL: {public_url}")
    print(f"  Password:   {PASSWORD}")
    print(f"{'='*60}")
else:
    print("ERROR: cloudflared URL を取得できませんでした")
    print("tunnel log:", ''.join(tunnel_lines[-10:]))

print("バックグラウンドで実行中")

## 8. ユーティリティ

ログ確認・サーバー停止用。

In [ ]:
# サーバーログ確認（直近20行）
!tail -20 /tmp/server.log

# サーバー + トンネル停止
server_proc.terminate()
tunnel_proc.terminate()
log_fh.close()
print("stopped")